### Chapter 2.6 - Probability and Statistics

Probability gives a language for uncertainty. Statistics gives tools for learning from data that varies.

Machine learning depends on both because data is noisy, samples are incomplete, and predictions are uncertain.


In [ ]:
import random
import numpy as np
import torch


### 2.6.1 A Simple Example: Tossing Coins

#### 1. Intuition

A random experiment is a process whose outcome is uncertain before it happens. Tossing a coin is the classic example.

An outcome is one possible result. For one coin toss, common outcomes are heads and tails.

Probability is a number between 0 and 1 that describes how likely an event is. An event is a set of outcomes we care about.

#### 2. Why this exists

ML systems often make predictions under uncertainty. Probability lets us model uncertainty instead of pretending every prediction is perfectly certain.


#### 3. Examples

Manual simulation of coin tosses with plain Python.


In [ ]:
outcomes = []
for _ in range(5):
    toss = random.choice(["H", "T"])
    outcomes.append(toss)
outcomes


Estimate the frequency of heads. Frequency means how often something happened in observed trials.


In [ ]:
heads = outcomes.count("H")
frequency = heads / len(outcomes)
frequency


Use PyTorch to sample 0 or 1 values, where 1 can represent heads.


In [ ]:
samples = torch.randint(0, 2, (5,))
heads_fraction = samples.float().mean()
samples, heads_fraction


#### 4. Step-by-step breakdown

`random.choice(["H", "T"])` selects one item randomly from the list.

The loop repeats the random experiment five times.

`outcomes.count("H")` counts heads.

`heads / len(outcomes)` computes the observed fraction of heads.

`torch.randint(0, 2, (5,))` samples integers from 0 up to but not including 2. So the possible values are 0 and 1.

`.float().mean()` converts integers to floating-point values and averages them.

#### 5. Connection to ML systems

Training data can be viewed as a sample from a larger population. A sample is the data we observed. A population is the broader collection we care about.

We train on samples, but we usually want performance on future unseen examples from the population.

#### 6. Common confusion points

- Probability is theoretical; frequency is observed.
- With few trials, observed frequency may differ a lot from the true probability.
- Random sampling produces different results unless the random seed is fixed.
- A probability of 0.5 does not guarantee exactly half heads in a small sample.


### 2.6.2 A More Formal Treatment

#### 1. Intuition

A sample space is the set of all possible outcomes of an experiment.

For one coin toss, the sample space is `{H, T}`. For one six-sided die roll, it is `{1, 2, 3, 4, 5, 6}`.

An event is a subset of the sample space. For a die, `roll is even` is the event `{2, 4, 6}`.

#### 2. Why this exists

Formal probability helps us reason consistently. Without clear sample spaces and events, probability statements become vague.


#### 3. Examples

Compute event probability for a fair die by counting outcomes.


In [ ]:
sample_space = [1, 2, 3, 4, 5, 6]
even = [x for x in sample_space if x % 2 == 0]
prob_even = len(even) / len(sample_space)
prob_even


Represent a probability distribution as values and probabilities.


In [ ]:
values = torch.tensor([1, 2, 3, 4])
probs = torch.tensor([0.1, 0.2, 0.3, 0.4])
probs.sum()


#### 4. Step-by-step breakdown

The die has six equally likely outcomes.

The event `even` contains three outcomes.

The probability is `3 / 6`, which equals 0.5.

A probability distribution assigns probabilities to possible values.

`probs.sum()` should be 1 for a complete discrete distribution. Discrete means the possible values are separate countable choices.

#### 5. Connection to ML systems

Classification models often output a probability distribution over classes. For example, an image classifier may assign probabilities to `cat`, `dog`, and `car`.

#### 6. Common confusion points

- A probability must be between 0 and 1.
- Probabilities over all mutually exclusive complete outcomes must sum to 1.
- An event can contain more than one outcome.
- Equal probability is an assumption, not automatic.


### 2.6.3 Random Variables

#### 1. Intuition

A random variable maps uncertain outcomes to numbers.

For a coin toss, define `X = 1` for heads and `X = 0` for tails. The coin outcome is text, but the random variable is numerical.

#### 2. Why this exists

ML needs numerical objects. Random variables let us describe uncertain data numerically, which makes computation possible.


#### 3. Examples

Manual mapping from outcomes to numbers.


In [ ]:
outcomes = ["H", "T", "H"]
X = [1 if outcome == "H" else 0 for outcome in outcomes]
X


A Bernoulli random variable has values 0 or 1. Bernoulli is the name for a yes/no random variable.


In [ ]:
p = torch.tensor(0.7)
samples = torch.bernoulli(p.repeat(5))
samples


#### 4. Step-by-step breakdown

The list comprehension checks each outcome.

If the outcome is heads, it stores 1. Otherwise, it stores 0.

`p = 0.7` means the probability of sampling 1 is 0.7.

`p.repeat(5)` creates five copies of that probability.

`torch.bernoulli(...)` samples 0 or 1 values using the supplied probabilities.

#### 5. Connection to ML systems

Labels in binary classification are often represented as Bernoulli-like values: 1 for positive class and 0 for negative class.

#### 6. Common confusion points

- A random variable is not the same as one sampled value.
- The name variable does not mean it changes after sampling; it means the value is uncertain before observation.
- Bernoulli variables only produce 0 or 1.
- The probability parameter controls long-run tendency, not each individual result with certainty.


### 2.6.4 Multiple Random Variables

#### 1. Intuition

Multiple random variables describe several uncertain quantities together.

A joint probability describes how two or more random variables behave together. A marginal probability describes one variable after ignoring the others. A conditional probability describes one variable when another is known.

#### 2. Why this exists

Real data has many variables. We often need to ask how one variable relates to another, such as whether a feature changes the probability of a label.


#### 3. Examples

Use a tiny table of paired observations.


In [ ]:
data = [("rain", "late"), ("sun", "on_time"), ("rain", "late")]
rain_count = sum(1 for weather, _ in data if weather == "rain")
late_count = sum(1 for _, status in data if status == "late")
rain_count / len(data), late_count / len(data)


Estimate a conditional probability: probability of late given rain.


In [ ]:
rain_days = [status for weather, status in data if weather == "rain"]
late_given_rain = sum(s == "late" for s in rain_days) / len(rain_days)
late_given_rain


#### 4. Step-by-step breakdown

Each tuple stores two observed variables: weather and arrival status.

The first calculation estimates marginal probabilities from counts.

`rain_days` filters the data to observations where weather was rain.

The conditional estimate counts how often status was late inside that filtered subset.

`P(late | rain)` is read as probability of late given rain.

#### 5. Connection to ML systems

Many ML models estimate relationships between inputs and labels. Conditional probability is a natural way to talk about prediction: what is the probability of a label given these features?

#### 6. Common confusion points

- Joint means together.
- Marginal means focusing on one variable while ignoring others.
- Conditional means after information is known.
- Correlation or association does not automatically prove causation.


### 2.6.5 An Example

#### 1. Intuition

A small bag-of-balls example makes joint and conditional probability concrete.

Suppose a bag contains balls with two properties: color and shape. We observe a few draws and estimate probabilities from counts.

#### 2. Why this exists

Examples show how probability language maps to data tables. This is close to how ML uses datasets: count, estimate, and predict under uncertainty.


#### 3. Examples

Create a tiny observed dataset.


In [ ]:
balls = [("red", "round"), ("red", "square"), ("blue", "round")]
red = sum(color == "red" for color, _ in balls)
round_shape = sum(shape == "round" for _, shape in balls)
red / len(balls), round_shape / len(balls)


Estimate probability of round shape given red color.


In [ ]:
red_balls = [shape for color, shape in balls if color == "red"]
p_round_given_red = sum(shape == "round" for shape in red_balls) / len(red_balls)
p_round_given_red


#### 4. Step-by-step breakdown

The dataset has three observations.

`red / len(balls)` estimates the marginal probability of red.

`round_shape / len(balls)` estimates the marginal probability of round.

Filtering to `red_balls` restricts attention to red observations.

The final line estimates `P(round | red)`.

#### 5. Connection to ML systems

A classifier estimates something similar: `P(class | features)`. The features may be pixels, words, or table columns. The class is the label to predict.

#### 6. Common confusion points

- Estimated probabilities from tiny data are unstable.
- Conditional probabilities depend on the condition set.
- If no observations match the condition, the estimate is undefined because division by zero would occur.
- Probability estimates improve with relevant data, not just with formulas.


### 2.6.6 Expectations

#### 1. Intuition

Expectation is the probability-weighted average value of a random variable.

For equally likely observed values, the sample mean estimates expectation. Variance measures how spread out values are around the mean.

#### 2. Why this exists

ML uses averages constantly. Loss is often averaged over examples. Statistics such as mean and variance describe data distributions and help normalize inputs.


#### 3. Examples

Manual mean and variance for a small list.


In [ ]:
values = [1.0, 2.0, 5.0]
mean = sum(values) / len(values)
variance = sum((v - mean) ** 2 for v in values) / len(values)
mean, variance


PyTorch mean and variance.


In [ ]:
x = torch.tensor([1.0, 2.0, 5.0])
mean = x.mean()
variance = ((x - mean) ** 2).mean()
mean, variance


Expectation from explicit values and probabilities.


In [ ]:
values = torch.tensor([0.0, 1.0])
probs = torch.tensor([0.25, 0.75])
expected = (values * probs).sum()
expected


#### 4. Step-by-step breakdown

The mean adds values and divides by how many values there are.

The variance first subtracts the mean from each value, squares the differences, and averages those squared differences.

`x - mean` uses broadcasting to subtract the scalar mean from every entry.

For explicit probabilities, expectation multiplies each value by its probability and sums the results.

#### 5. Connection to ML systems

Expected loss is the average loss we would see over the data distribution. Training usually approximates it with an average over available examples.

#### 6. Common confusion points

- Expectation is a theoretical average; sample mean is an estimate from data.
- Variance is about spread, not central value.
- Squaring makes negative and positive deviations both contribute positively.
- Averages can hide subgroups with very different behavior.


### 2.6.7 Discussion

#### 1. Intuition

Probability and statistics help us reason about uncertainty, limited data, and variation.

Probability starts from a model of randomness. Statistics starts from observed data and estimates useful quantities.

#### 2. Why this exists

Machine learning sits between both views. We observe data, estimate patterns, and build models that make uncertain predictions about new cases.


#### 3. Examples

A tiny empirical risk calculation. Risk here means average loss over examples.


In [ ]:
pred = torch.tensor([0.2, 0.7, 0.9])
y = torch.tensor([0.0, 1.0, 1.0])
losses = (pred - y) ** 2
risk = losses.mean()
risk


#### 4. Step-by-step breakdown

`pred` stores model predictions.

`y` stores labels.

`(pred - y) ** 2` computes squared error for each example.

`losses.mean()` averages the losses into one scalar estimate.

This average is empirical because it is computed from observed data.

#### 5. Connection to ML systems

Training minimizes empirical loss, hoping it also reduces loss on future unseen data. This hope depends on the training data being informative about the future data.

#### 6. Common confusion points

- Training data is a sample, not the whole world.
- Randomness can come from data collection, initialization, batching, or hardware behavior.
- A low average loss may still hide rare but important failures.
- Probability language should be tied to a clear sample space or data-generating process.


### 2.6.8 Exercises

#### 1. Intuition

These exercises practice probability as counting, simulation, and averaging.

#### 2. Why this exists

Small probability exercises build the habits needed to understand losses, prediction confidence, and evaluation metrics.


#### 3. Examples

Exercise 1: Estimate the probability of rolling at least 5 on a fair die by counting outcomes.


In [ ]:
die = [1, 2, 3, 4, 5, 6]
event = [x for x in die if x >= 5]
len(event) / len(die)


Exercise 2: Simulate five Bernoulli samples with probability 0.6.


In [ ]:
p = torch.tensor(0.6)
samples = torch.bernoulli(p.repeat(5))
samples


Exercise 3: Compute mean and variance of a tiny tensor.


In [ ]:
x = torch.tensor([2.0, 4.0, 4.0])
mean = x.mean()
variance = ((x - mean) ** 2).mean()
mean, variance


Exercise 4: Estimate `P(round | blue)` from observations.


In [ ]:
items = [("blue", "round"), ("red", "square"), ("blue", "square")]
blue_shapes = [shape for color, shape in items if color == "blue"]
sum(shape == "round" for shape in blue_shapes) / len(blue_shapes)


#### 4. Step-by-step breakdown

Exercise 1 uses formal counting.

Exercise 2 uses random simulation.

Exercise 3 computes statistics from values.

Exercise 4 computes conditional probability by filtering observations.

#### 5. Connection to ML systems

These ideas reappear in classification, loss functions, uncertainty, and evaluation.

#### 6. Common confusion points

- Distinguish exact probabilities from estimates.
- Small samples can be misleading.
- Conditional probability requires a condition group with at least one observation.
- Mean and variance summarize different aspects of data.
